In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


class DenseNet201(nn.Module):
  """Wraps a pre-trained torchvision DenseNet model to extract both raw

  DenseBlock outputs and pre-ReLU transition features for ReviewKD.
  preact_feats: [stem, f1_pre, f2_pre, f3_pre, f4_pre] representing the pre-ReLU features from the stem and each of the four DenseBlocks.
  earlier_feats: [earlier_feat1, earlier_feat2, earlier_feat3, earlier_feat4] representing eariler feature maps of DenseNet's stage.
  """

  def __init__(self, model: nn.Module = models.densenet201(weights=None)):
    super().__init__()
    self.model = model
    self.features = model.features
    self.classifier = model.classifier

    # Output channels for DenseNet-201: [56x56, 28x28, 14x14, 7x7, 1x1]
    self.stage_channels = [256, 512, 1792, 1920, 1920]

  def get_bn_before_relu(self):
    return [
        self.features.transition1.norm,
        self.features.transition2.norm,
        self.features.transition3.norm,
        self.features.norm5,
    ]

  def get_stage_channels(self):
    return self.stage_channels

  def forward(self, x):
    # Stem: 224x224 -> 56x56
    x = self.features.conv0(x) # input size from 3x224x224 to 64x112x112
    x = self.features.norm0(x) # input size from 64x112x112 to 64x112x112
    stem = x
    x = self.features.relu0(x) # input size from 64x112x112 to 64x112x112
    x = self.features.pool0(x) # input size from 64x112x112 to 64x56x56
    earlier_feat1 = x

    # Stage 1 (56x56)
    db1_out = self.features.denseblock1(x)  # Raw DenseBlock 1
    f1_pre = self.features.transition1.norm(
        db1_out
    )  # Pre-ReLU normalized feature
    x = self.features.transition1.pool(
        self.features.transition1.conv(self.features.transition1.relu(f1_pre))
    )
    earlier_feat2 = x

    # Stage 2 (28x28)
    db2_out = self.features.denseblock2(x)  # Raw DenseBlock 2
    f2_pre = self.features.transition2.norm(
        db2_out
    )  # Pre-ReLU normalized feature
    x = self.features.transition2.pool(
        self.features.transition2.conv(self.features.transition2.relu(f2_pre))
    )
    earlier_feat3 = x

    # Stage 3 (14x14)
    db3_out = self.features.denseblock3(x)  # Raw DenseBlock 3
    f3_pre = self.features.transition3.norm(
        db3_out
    )  # Pre-ReLU normalized feature
    x = self.features.transition3.pool(
        self.features.transition3.conv(self.features.transition3.relu(f3_pre))
    )
    earlier_feat4 = x

    # Stage 4 (7x7)
    db4_out = self.features.denseblock4(x)  # Raw DenseBlock 4
    f4_pre = self.features.norm5(db4_out)  # Pre-ReLU normalized feature
    f4 = F.relu(f4_pre, inplace=True)

    # Stage 5 (1x1 GAP)
    f5_gap = F.adaptive_avg_pool2d(f4, (1, 1))  # (B, 1920, 1, 1)
    pooled = f5_gap.flatten(1) # shape: from (B, 1920, 1, 1) to (B, 1920)
    out = self.classifier(pooled)

    feats = {
        "denseblock_feats": [db1_out, db2_out, db3_out, db4_out],
        "earlier_feats": [earlier_feat1, earlier_feat2, earlier_feat3, earlier_feat4],
        "preact_feats": [stem, f1_pre, f2_pre, f3_pre, f4_pre],
        "pooled_feat": pooled,
    }

    return out, feats